# UniProt REST API Lesson

This notebook shows how to use `yanglab.UniProt` with the newer UniProt REST endpoint.

We will move through four ideas:

1. Fetch one UniProt record as raw JSON
2. Inspect the top-level fields in the response
3. Parse the record with `UniprotRecord`

## 1. Setup

We first move into the workshop directory and import the UniProt helpers.

In [2]:
import os
os.chdir('..')

from yanglab.UniProt import queryUniprot, searchUniprot


## 2. Fetch a UniProt record as raw JSON

A good starting point is a UniProt accession. Here we use `P04439`.

`queryUniprot()` returns the raw JSON dictionary from the UniProt REST API.

In [3]:
accession = 'P04439'
raw = queryUniprot(accession)
type(raw), raw['primaryAccession']


(dict, 'P04439')

## 3. Inspect the shape of the JSON

Before parsing deeply, it is always useful to inspect the top-level keys.

In [4]:
sorted(raw.keys())


['annotationScore',
 'comments',
 'entryAudit',
 'entryType',
 'extraAttributes',
 'features',
 'genes',
 'keywords',
 'organism',
 'primaryAccession',
 'proteinDescription',
 'proteinExistence',
 'references',
 'secondaryAccessions',
 'sequence',
 'uniProtKBCrossReferences',
 'uniProtkbId']

In [5]:
raw['proteinDescription']


{'recommendedName': {'fullName': {'value': 'HLA class I histocompatibility antigen, A alpha chain'}},
 'alternativeNames': [{'fullName': {'value': 'Human leukocyte antigen A'},
   'shortNames': [{'value': 'HLA-A'}]}],
 'flag': 'Precursor'}

In [6]:
raw['organism']


{'scientificName': 'Homo sapiens',
 'commonName': 'Human',
 'taxonId': 9606,
 'lineage': ['Eukaryota',
  'Metazoa',
  'Chordata',
  'Craniata',
  'Vertebrata',
  'Euteleostomi',
  'Mammalia',
  'Eutheria',
  'Euarchontoglires',
  'Primates',
  'Haplorrhini',
  'Catarrhini',
  'Hominidae',
  'Homo']}

## 4. Parse the record with `UniprotRecord`

`searchUniprot()` fetches the record and wraps it inside a `UniprotRecord` object.
This makes the data easier to use through biologically meaningful getter methods.

In [7]:
record = searchUniprot(accession)
record


@> Parse UniProt information of P04439...
@> Parsing in 0.0s.


<UniprotRecord: P04439 (HLAA_HUMAN)>

In [8]:
record.getOrganism()

{'scientific_name': 'Homo sapiens',
 'common_name': 'Human',
 'taxonomy_id': 9606,
 'lineage': ['Eukaryota',
  'Metazoa',
  'Chordata',
  'Craniata',
  'Vertebrata',
  'Euteleostomi',
  'Mammalia',
  'Eutheria',
  'Euarchontoglires',
  'Primates',
  'Haplorrhini',
  'Catarrhini',
  'Hominidae',
  'Homo']}

## 5. Basic biological fields

These are usually the first fields we want from UniProt.

In [9]:
record.getAccession()


'P04439'

In [10]:
record.getName()


'HLAA_HUMAN'

In [11]:
record.getProtein()


{'recommend_name': 'HLA class I histocompatibility antigen, A alpha chain',
 'alter_fullname': 'Human leukocyte antigen A',
 'alter_shortname': 'HLA-A'}

In [12]:
record.getGene()


'HLA-A'

In [13]:
record.getOrganism()


{'scientific_name': 'Homo sapiens',
 'common_name': 'Human',
 'taxonomy_id': 9606,
 'lineage': ['Eukaryota',
  'Metazoa',
  'Chordata',
  'Craniata',
  'Vertebrata',
  'Euteleostomi',
  'Mammalia',
  'Eutheria',
  'Euarchontoglires',
  'Primates',
  'Haplorrhini',
  'Catarrhini',
  'Hominidae',
  'Homo']}

In [14]:
sequence = record.getSequence()
len(sequence), sequence[:60]


(365, 'MAVMAPRTLLLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRF')

## 6. Features and comments

UniProt records contain two very important blocks:

- `features`: positional annotations on the sequence
- `comments`: descriptive annotations such as subunit, PTM, and interactions

In [15]:
len(record.getFeatures()), len(record.getComments())


(162, 33)

In [16]:
record.getBindingSite()[:3]


[{'position': 31,
  'description': '',
  'name': 'a peptide antigen',
  'chebi': 'CHEBI:166823'},
 {'position': 97,
  'description': '',
  'name': 'a peptide antigen',
  'chebi': 'CHEBI:166823'},
 {'position': 108,
  'description': '',
  'name': 'a peptide antigen',
  'chebi': 'CHEBI:166823'}]

In [17]:
record.getActiveSite()[:3]


[]

In [18]:
record.getPTMProcessing()[:5]


[{'type': 'Signal', 'begin': 1, 'end': 24, 'description': ''},
 {'type': 'Chain',
  'begin': 25,
  'end': 365,
  'description': 'HLA class I histocompatibility antigen, A alpha chain'},
 {'type': 'Modified residue', 'position': 83, 'description': 'Sulfotyrosine'},
 {'type': 'Modified residue', 'position': 343, 'description': 'Phosphoserine'},
 {'type': 'Modified residue',
  'position': 344,
  'description': 'Phosphotyrosine'}]

In [19]:
record.getSubunit('all')[:3]


['Heterotrimer that consists of an alpha chain HLA-A, a beta chain B2M and a peptide (peptide-HLA-A-B2M) (PubMed:11502003, PubMed:18275829, PubMed:19177349, PubMed:19542454, PubMed:21943705, PubMed:22245737, PubMed:24395804, PubMed:26758806, PubMed:28250417, PubMed:7504010, PubMed:7506728, PubMed:7679507, PubMed:7694806, PubMed:7935798, PubMed:8805302, PubMed:8906788, PubMed:9177355). Early in biogenesis, HLA-A-B2M dimer interacts with the components of the peptide-loading complex composed of TAPBP, TAP1-TAP2, TAPBPL, PDIA3/ERP57 and CALR (PubMed:21263072). Interacts with TAP1-TAP2 transporter via TAPBP; this interaction is obligatory for the loading of peptide epitopes delivered to the ER by TAP1-TAP2 transporter (PubMed:21263072, PubMed:8630735, PubMed:8805302). Interacts with TAPBPL; TAPBPL binds peptide-free HLA-A-B2M complexes or those loaded with low affinity peptides, likely facilitating peptide exchange for higher affinity peptides (PubMed:26869717, PubMed:35725941). Only optim

In [26]:
record.getPDBs()

{'1AKJ': {'method': 'X-ray',
  'resolution': 2.65,
  'chains': ['A'],
  'resrange': '25-300'},
 '1AO7': {'method': 'X-ray',
  'resolution': 2.6,
  'chains': ['A'],
  'resrange': '25-299'},
 '1AQD': {'method': 'X-ray',
  'resolution': 2.45,
  'chains': ['C', 'F', 'I', 'L'],
  'resrange': '127-141'},
 '1B0G': {'method': 'X-ray',
  'resolution': 2.5,
  'chains': ['A', 'D'],
  'resrange': '25-299'},
 '1B0R': {'method': 'X-ray',
  'resolution': 2.9,
  'chains': ['A'],
  'resrange': '25-299'},
 '1BD2': {'method': 'X-ray',
  'resolution': 2.5,
  'chains': ['A'],
  'resrange': '25-299'},
 '1DUY': {'method': 'X-ray',
  'resolution': 2.15,
  'chains': ['A', 'D'],
  'resrange': '25-299'},
 '1DUZ': {'method': 'X-ray',
  'resolution': 1.8,
  'chains': ['A', 'D'],
  'resrange': '25-299'},
 '1EEY': {'method': 'X-ray',
  'resolution': 2.25,
  'chains': ['A', 'D'],
  'resrange': '25-299'},
 '1EEZ': {'method': 'X-ray',
  'resolution': 2.3,
  'chains': ['A', 'D'],
  'resrange': '25-299'},
 '1HHG': {'meth

## 7. Cross-references to other databases

UniProt links one protein record to many other resources.
These are available through cross-references.

In [20]:
sorted(record.getCrossReferences().keys())[:20]


['ABCD',
 'AGR',
 'Agora',
 'AlphaFoldDB',
 'Antibodypedia',
 'Bgee',
 'BindingDB',
 'BioGRID',
 'BioGRID-ORCS',
 'BioMuta',
 'CCDS',
 'CDD',
 'CIViC',
 'CPTC',
 'CTD',
 'ChEMBL',
 'ChiTaRS',
 'ClinPGx',
 'ComplexPortal',
 'DMDM']

In [21]:
record.getAlphaFold()


'P04439'

In [22]:
record.getComplexPortals()


[{'id': 'CPX-26559', 'name': 'Classical MHC Ia complex, HLA-A-B2M'}]

In [23]:
list(record.getPDBs().items())[:3]


[('1AKJ',
  {'method': 'X-ray',
   'resolution': 2.65,
   'chains': ['A'],
   'resrange': '25-300'}),
 ('1AO7',
  {'method': 'X-ray',
   'resolution': 2.6,
   'chains': ['A'],
   'resrange': '25-299'}),
 ('1AQD',
  {'method': 'X-ray',
   'resolution': 2.45,
   'chains': ['C', 'F', 'I', 'L'],
   'resrange': '127-141'})]

## 8. Keywords and references

These are useful for getting a compact summary of function and literature.

In [24]:
record.getKeywords()[:10]


['3D-structure',
 'Adaptive immunity',
 'Alternative splicing',
 'Cell membrane',
 'Direct protein sequencing',
 'Disulfide bond',
 'Endoplasmic reticulum',
 'Glycoprotein',
 'Host-virus interaction',
 'Immunity']

In [25]:
record.getReferences()[:2]


[{'id': None,
  'type': 'journal article',
  'title': 'The primary structure of HLA-A32 suggests a region involved in formation of the Bw4/Bw6 epitopes.',
  'journal': 'J. Immunol.',
  'volume': '137',
  'first_page': '3671',
  'last_page': '3674',
  'year': '1986',
  'authors': ['Wan A.M.', 'Ennis P.', 'Parham P.', 'Holmes N.']},
 {'id': None,
  'type': 'journal article',
  'title': 'Multiple genetic mechanisms have contributed to the generation of the HLA-A2/A28 family of class I MHC molecules.',
  'journal': 'J. Immunol.',
  'volume': '139',
  'first_page': '936',
  'last_page': '941',
  'year': '1987',
  'authors': ['Holmes N.',
   'Ennis P.',
   'Wan A.M.',
   'Denney D.W.',
   'Parham P.']}]

## 9. Teaching takeaway

The main workflow is:

1. fetch the UniProt REST JSON
2. inspect the response structure
3. parse it into a Python object
4. extract the biological fields you care about